# GLiNER2 Biography Extraction Pipeline

This notebook implements the data extraction and processing pipeline using **GLiNER2**, the unified successor to the original GLiNER library. 

## 1. Setup and Installation

In [39]:
!pip install -q gliner2 tqdm pandas

## 2. Dataset Loading

In [40]:
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from gliner2 import GLiNER2
from collections import defaultdict

def load_biographies(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

# User specified Kaggle Path
KAGLLE_INPUT_PATH = '/kaggle/input/datasets/pallamreddyviswas/train-set-wikibio/openie_input.txt'

## 3. GLiNER2 Implementation

In [41]:
class GLiNER2Pipeline:
    def __init__(self, model_name="fastino/gliner2-base-v1"):
        print(f"Loading model: {model_name}...")
        self.model = GLiNER2.from_pretrained(model_name)
        
        # Defined labels for extraction
        self.labels = [
            "person", "occupation", "education", 
            "award", "notable_work", "birth_date", "birth_place",
            "nationality", "organization"
        ]

    def extract(self, texts, batch_size=16):
        results = []
        for i in tqdm(range(0, len(texts), batch_size), desc="GLiNER2 Inference"):
            batch = texts[i:i+batch_size]
            try:
                 batch_results = self.model.extract_entities(batch, self.labels)
                 results.extend(batch_results)
            except Exception as e:
                 for text in batch:
                     res = self.model.extract_entities(text, self.labels)
                     results.append(res)
        return results

def process_and_save(input_file, output_file, limit=None):
    print(f"Reading input: {input_file}")
    bios = load_biographies(input_file)
    if limit:
        bios = bios[:limit]
        
    pipeline = GLiNER2Pipeline()
    extracted_data = pipeline.extract(bios)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(extracted_data, f, indent=2)
    
    print(f"Done! Saved {len(extracted_data)} records to {output_file}")
    return extracted_data

# --- EXECUTION ---
extracted_data = process_and_save(
    input_file=KAGLLE_INPUT_PATH, 
    output_file='/kaggle/working/extracted_biographies.json'
)

Reading input: /kaggle/input/datasets/pallamreddyviswas/train-set-wikibio/openie_input.txt
Loading model: fastino/gliner2-base-v1...


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


GLiNER2 Inference:   0%|          | 0/125 [00:00<?, ?it/s]

Done! Saved 2000 records to /kaggle/working/extracted_biographies.json


## 4. Knowledge Graph Construction
We convert the structured JSON output into a (Subject, Relation, Object) graph.

In [42]:
def json_to_triples(extracted_list):
    """
    Converts GLiNER2 output list into triples.
    Each entry is expected to be a dict with key 'entities'
    """
    triples = []
    mapping = {
        "occupation": "has_occupation",
        "education": "studied_at",
        "award": "received_award",
        "notable_work": "created_work",
        "birth_place": "born_in",
        "organization": "linked_to"
    }

    for entry in extracted_list:
        entities = entry.get("entities", {})
        
        # Identify the main subject (usually the first 'person' found)
        persons = entities.get("person", [])
        if not persons:
            continue
        subject = persons[0]

        for entity_type, relation in mapping.items():
            for obj in entities.get(entity_type, []):
                triples.append((subject, relation, obj))
    return triples

def build_kg(triples):
    kg = defaultdict(lambda: defaultdict(list))
    for s, r, o in triples:
        if o not in kg[s][r]:
            kg[s][r].append(o)
    return kg

# execution
triples = json_to_triples(extracted_data)
kg = build_kg(triples)

def preview_kg(kg, num_subjects=5):
    for i, (subj, rels) in enumerate(list(kg.items())[:num_subjects]):
        print(f"\n[Subject]: {subj}")
        for r, objs in rels.items():
            print(f"  ├── {r} -> {', '.join(objs)}")

preview_kg(kg)


[Subject]: walter extra
  ├── has_occupation -> aerobatic pilot, chief aircraft designer
  ├── studied_at -> mechanical engineer
  ├── received_award -> 1982 world aerobatic championships
  ├── created_work -> extra ea-230, pitts special aircraft, unlimited aerobatic aircraft, turboprop transports
  ├── linked_to -> extra flugzeugbau, extra firm

[Subject]: aaron hohlbein
  ├── has_occupation -> soccer player
  ├── born_in -> middleton, wisconsin
  ├── linked_to -> club

[Subject]: majda vrhovnik
  ├── has_occupation -> medical student
  ├── received_award -> people 's hero of yugoslavia
  ├── born_in -> klagenfurt
  ├── linked_to -> communist party of slovenia

[Subject]: linda hayden
  ├── has_occupation -> actress
  ├── created_work -> sex comedies, 1970s british horror films

[Subject]: craig starcevich
  ├── has_occupation -> fitness trainer, strength and conditioning coach
  ├── received_award -> 1986 f. d. book medal, best and fairest player
  ├── born_in -> east perth
  ├── li

## 5. Quality Inspection
Let's look at a few random samples to check extraction precision.

In [43]:
import random

def check_quality(data, num_samples=3):
    samples = random.sample(data, min(len(data), num_samples))
    for i, sample in enumerate(samples):
        print(f"\n--- Sample {i+1} ---")
        # Note: We need a way to link back to original text for quality check,
        # usually GLiNER returns entities with spans. 
        print(json.dumps(sample, indent=2))

check_quality(extracted_data)


--- Sample 1 ---
{
  "entities": {
    "person": [
      "judith stephenson",
      "luther scott harshbarger"
    ],
    "occupation": [
      "lawyer",
      "senior counsel",
      "politician"
    ],
    "education": [],
    "award": [],
    "notable_work": [],
    "birth_date": [
      "december 1, 1941"
    ],
    "birth_place": [
      "commonwealth of massachusetts",
      "boston"
    ],
    "nationality": [],
    "organization": [
      "proskauer rose"
    ]
  }
}

--- Sample 2 ---
{
  "entities": {
    "person": [
      "anthony andrew `` tony '' jarvis"
    ],
    "occupation": [
      "swimmer"
    ],
    "education": [
      "university of massachusetts",
      "london university",
      "master of business administration"
    ],
    "award": [],
    "notable_work": [],
    "birth_date": [
      "3 march 1945"
    ],
    "birth_place": [
      "toronto, canada"
    ],
    "nationality": [
      "british"
    ],
    "organization": [
      "ctn media group",
      "millw